# JWST ZTF J1539 — Flux Over Time Plotter
Plots photon flux (`flux_recalc`) vs time (`int_mid_BJD_TDB`) from the JWST crfints.fits files.

In [ ]:
from astropy.io import fits
from astropy.time import Time
from astropy import units as u
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ✏️ Set your data path
original_path = Path(r"T:\MAST_2023-10-07T0548\JWST")

# ✏️ Choose what to plot
EPOCH     = "epoch1"   # "epoch1" or "epoch2"
WAVE_TYPE = "sw"       # "sw" (F070W) or "lw" (F277W / F322W2)
TIME_UNIT = "minute"   # "second", "minute", "hour", or "day"

## Step 1 — Helper: convert time units

In [ ]:
def convert_time_units(mjd_times, unit):
    """Convert MJD array to elapsed time in chosen unit (relative to first frame)."""
    t = Time(mjd_times, format="mjd", scale="tdb")
    factors = {"second": 86400.0, "minute": 1440.0, "hour": 24.0, "day": 1.0}
    if unit not in factors:
        raise ValueError(f"Unknown unit '{unit}'. Choose: second, minute, hour, day")
    return (t.mjd - t.mjd.min()) * factors[unit]

## Step 2 — Load all files and extract flux + time per frame

In [ ]:
# Select file pattern based on epoch and wave type
patterns = {
    ("epoch1", "sw"): "jw01666001001*_nrcb1/*crfints.fits",
    ("epoch1", "lw"): "jw01666001001*_nrcblong/*crfints.fits",
    ("epoch2", "sw"): "jw01666003001*_nrcb1/*crfints.fits",
    ("epoch2", "lw"): "jw01666003001*_nrcblong/*crfints.fits",
}

file_list = sorted(original_path.glob(patterns[(EPOCH, WAVE_TYPE)]))
print(f"Found {len(file_list)} file(s) for {EPOCH} / {WAVE_TYPE}:")
for f in file_list:
    print(f"  {f.name}")

In [ ]:
all_times_mjd = []   # mid-integration time in MJD (BJD TDB)
all_flux      = []   # summed flux per frame (MJy/sr * pixels)
all_flux_err  = []   # propagated uncertainty
all_frames    = []   # frame index
all_filenames = []   # source filename

for filename in file_list:
    print(f"\nProcessing: {filename.name}")

    with fits.open(filename) as f:
        sci  = f["SCI"].data          # shape: (n_frames, 160, 160) — flux in MJy/sr
        err  = f["ERR"].data          # shape: (n_frames, 160, 160) — uncertainty
        dq   = f["DQ"].data           # shape: (n_frames, 160, 160) — data quality flags
        times_table = f["INT_TIMES"].data  # per-integration timing table

        n_frames = sci.shape[0]
        fname    = f[0].header["FILENAME"]

        # ✏️ Adjust target pixel position for your source
        # These are the (x, y) pixel coords of ZTF J1539 in the subarray
        centers = {"F070W": (83, 85), "F277W": (86, 76), "F322W2": (86, 76)}
        filt    = f[0].header["FILTER"]
        r_ap    = {"F070W": 3, "F277W": 2, "F322W2": 3}  # aperture radius in pixels
        xc, yc  = centers[filt]
        r       = r_ap[filt]

        # Build a circular aperture mask
        ny, nx = sci.shape[1], sci.shape[2]
        yy, xx = np.ogrid[:ny, :nx]
        ap_mask = (xx - xc)**2 + (yy - yc)**2 <= r**2  # True inside aperture

        for i in range(n_frames):  # ✅ range(n_frames), NOT range(n_frames+1)
            frame_data = sci[i]
            frame_err  = err[i]
            frame_dq   = dq[i]

            # Mask bad pixels (DQ > 0 means flagged)
            good = ap_mask & (frame_dq == 0)

            if good.sum() == 0:
                print(f"  Frame {i+1}: all pixels flagged, skipping")
                continue

            # Simple aperture sum (background subtraction can be added later)
            flux     = np.nansum(frame_data[good])
            flux_err = np.sqrt(np.nansum(frame_err[good]**2))

            # Time: mid-integration BJD TDB
            t_mjd = times_table["int_mid_BJD_TDB"][i]

            all_times_mjd.append(t_mjd)
            all_flux.append(flux)
            all_flux_err.append(flux_err)
            all_frames.append(i + 1)
            all_filenames.append(fname)

# Convert to numpy arrays
all_times_mjd = np.array(all_times_mjd)
all_flux      = np.array(all_flux)
all_flux_err  = np.array(all_flux_err)
all_frames    = np.array(all_frames)

print(f"\nTotal frames extracted: {len(all_times_mjd)}")

## Step 3 — Convert time to chosen unit and plot

In [ ]:
# Convert MJD → elapsed time in chosen unit
time_elapsed = convert_time_units(all_times_mjd, TIME_UNIT)

xlabel_map = {
    "second": "Elapsed Time (seconds)",
    "minute": "Elapsed Time (minutes)",
    "hour":   "Elapsed Time (hours)",
    "day":    "Elapsed Time (days)",
}

filter_label = {"sw": "F070W (SW)", "lw": "F277W/F322W2 (LW)"}[WAVE_TYPE]
title = f"ZTF J1539 — Flux vs Time\n{EPOCH.capitalize()}  |  {filter_label}"

fig = plt.figure(figsize=(14, 8))
gs  = gridspec.GridSpec(2, 1, height_ratios=[3, 1], hspace=0.08)

# --- Main flux plot ---
ax1 = fig.add_subplot(gs[0])
ax1.errorbar(
    time_elapsed, all_flux, yerr=all_flux_err,
    fmt='o', ms=2.5, color='steelblue', ecolor='lightblue',
    elinewidth=0.8, capsize=0, alpha=0.8, label="Aperture flux"
)
ax1.set_ylabel("Flux (MJy/sr × pixels)", fontsize=12)
ax1.set_title(title, fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.set_xticklabels([])  # hide x labels on top panel

# Highlight 5-sigma outliers
median_flux = np.nanmedian(all_flux)
std_flux    = np.nanstd(all_flux)
outliers    = np.abs(all_flux - median_flux) > 5 * std_flux
if outliers.sum() > 0:
    ax1.scatter(time_elapsed[outliers], all_flux[outliers],
                color='red', s=20, zorder=5, label=f"{outliers.sum()} outliers (>5σ)")
    ax1.legend(fontsize=10)

# --- SNR panel ---
ax2 = fig.add_subplot(gs[1], sharex=ax1)
snr = all_flux / np.where(all_flux_err > 0, all_flux_err, np.nan)
ax2.plot(time_elapsed, snr, '.', ms=2, color='darkorange', alpha=0.7)
ax2.axhline(3, color='red', linestyle='--', linewidth=0.8, label='SNR = 3')
ax2.set_xlabel(xlabel_map[TIME_UNIT], fontsize=12)
ax2.set_ylabel("SNR", fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, linestyle='--')

plt.savefig(f"flux_vs_time_{EPOCH}_{WAVE_TYPE}.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: flux_vs_time_{EPOCH}_{WAVE_TYPE}.png")

## Step 4 — Quick statistics

In [ ]:
print(f"{'='*45}")
print(f"  Summary: {EPOCH} / {WAVE_TYPE}")
print(f"{'='*45}")
print(f"  Total frames:      {len(all_flux)}")
print(f"  Time span:         {time_elapsed.max():.2f} {TIME_UNIT}s")
print(f"  Median flux:       {np.nanmedian(all_flux):.4f} MJy/sr·pix")
print(f"  Flux std dev:      {np.nanstd(all_flux):.4f} MJy/sr·pix")
print(f"  Median SNR:        {np.nanmedian(snr):.1f}")
print(f"  Frames SNR < 3:    {(snr < 3).sum()}")
print(f"  5σ outliers:       {outliers.sum()}")
print(f"{'='*45}")

## Step 5 — Plot all epochs and wave types together (overview)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 9), sharey=False)
fig.suptitle("ZTF J1539 — Flux vs Time (All Epochs & Filters)", fontsize=14, fontweight='bold')

colors = {"sw": "steelblue", "lw": "tomato"}

for row, epoch in enumerate(["epoch1", "epoch2"]):
    for col, wave in enumerate(["sw", "lw"]):
        ax = axes[row][col]
        pattern = patterns[(epoch, wave)]
        files = sorted(original_path.glob(pattern))

        t_all, f_all, fe_all = [], [], []

        for filename in files:
            try:
                with fits.open(filename) as hdul:
                    sci  = hdul["SCI"].data
                    err  = hdul["ERR"].data
                    dq   = hdul["DQ"].data
                    times_table = hdul["INT_TIMES"].data
                    filt = hdul[0].header["FILTER"]
                    xc, yc = centers[filt]
                    r = r_ap[filt]
                    ny, nx = sci.shape[1], sci.shape[2]
                    yy, xx = np.ogrid[:ny, :nx]
                    ap_mask = (xx - xc)**2 + (yy - yc)**2 <= r**2
                    for i in range(sci.shape[0]):
                        good = ap_mask & (dq[i] == 0)
                        if good.sum() == 0:
                            continue
                        t_all.append(times_table["int_mid_BJD_TDB"][i])
                        f_all.append(np.nansum(sci[i][good]))
                        fe_all.append(np.sqrt(np.nansum(err[i][good]**2)))
            except Exception as e:
                print(f"Skipping {filename.name}: {e}")

        if len(t_all) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        else:
            t_arr = np.array(t_all)
            f_arr = np.array(f_all)
            fe_arr = np.array(fe_all)
            t_elapsed = convert_time_units(t_arr, TIME_UNIT)
            ax.errorbar(t_elapsed, f_arr, yerr=fe_arr,
                        fmt='.', ms=2, color=colors[wave],
                        ecolor='lightgray', elinewidth=0.5, alpha=0.7)

        label = {"sw": "F070W (SW)", "lw": "F277W/F322W2 (LW)"}[wave]
        ax.set_title(f"{epoch.capitalize()} — {label}", fontsize=11)
        ax.set_xlabel(xlabel_map[TIME_UNIT], fontsize=9)
        ax.set_ylabel("Flux (MJy/sr·pix)", fontsize=9)
        ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig("flux_vs_time_all.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: flux_vs_time_all.png")